In [19]:

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight
from category_encoders import TargetEncoder
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import random

# Fix seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

df=pd.read_csv("preprocessed_outlier_median_data.csv")

X_scaled=df.drop(["Target"],axis=1)
y_cleaned = df["Target"]
le=LabelEncoder()
le.fit(y_cleaned)
# print("")

LabelEncoder()

In [18]:
# Define nn model
class SimpleNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.sigmoid = nn.Sigmoid()
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.sigmoid(x)
        x = self.fc2(x)
        return x

# Stratified K-Fold setup
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop
fold = 1
best_overall_acc = 0
best_preds, best_labels = None, None

for train_idx, val_idx in skf.split(X_scaled, y_cleaned):
    print(f"\n--- Fold {fold} ---")
    x_train_fold, x_val_fold = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train_fold, y_val_fold = y_cleaned.iloc[train_idx], y_cleaned.iloc[val_idx]

    smote = SMOTE(random_state=42)
    x_train_res, y_train_res = smote.fit_resample(x_train_fold, y_train_fold)

    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_res), y=y_train_res)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(torch.tensor(x_train_res.values, dtype=torch.float32), torch.tensor(y_train_res.values, dtype=torch.long)), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(x_val_fold.values, dtype=torch.float32), torch.tensor(y_val_fold.values, dtype=torch.long)), batch_size=64)

    model = SimpleNN(input_dim=X_scaled.shape[1], output_dim=len(np.unique(y_cleaned))).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader), epochs=25)

    best_fold_acc = 0
    best_fold_preds, best_fold_labels = None, None

    for epoch in range(25):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                out = model(xb)
                all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                all_labels.extend(yb.numpy())

        acc = np.mean(np.array(all_preds) == np.array(all_labels))
        if acc > best_fold_acc:
            best_fold_acc = acc
            best_fold_preds = all_preds
            best_fold_labels = all_labels

        print(f"Epoch {epoch+1:2d} | Fold {fold} | Val Accuracy: {acc:.4f}")

    if best_fold_acc > best_overall_acc:
        best_overall_acc = best_fold_acc
        best_preds = best_fold_preds
        best_labels = best_fold_labels

    fold += 1
# target_names = sorted(y_cleaned.unique().tolist())
print("\nBest Validation Accuracy across all folds:", round(best_overall_acc, 4))
print("\nClassification Report:")
target_names = [str(cls) for cls in le.classes_]
print(classification_report(best_labels, best_preds, target_names=target_names))




--- Fold 1 ---
Epoch  1 | Fold 1 | Val Accuracy: 0.3657
Epoch  2 | Fold 1 | Val Accuracy: 0.4459
Epoch  3 | Fold 1 | Val Accuracy: 0.5711
Epoch  4 | Fold 1 | Val Accuracy: 0.6348
Epoch  5 | Fold 1 | Val Accuracy: 0.6744
Epoch  6 | Fold 1 | Val Accuracy: 0.6813
Epoch  7 | Fold 1 | Val Accuracy: 0.7081
Epoch  8 | Fold 1 | Val Accuracy: 0.7148
Epoch  9 | Fold 1 | Val Accuracy: 0.7383
Epoch 10 | Fold 1 | Val Accuracy: 0.7460
Epoch 11 | Fold 1 | Val Accuracy: 0.7566
Epoch 12 | Fold 1 | Val Accuracy: 0.7655
Epoch 13 | Fold 1 | Val Accuracy: 0.7726
Epoch 14 | Fold 1 | Val Accuracy: 0.7828
Epoch 15 | Fold 1 | Val Accuracy: 0.7856
Epoch 16 | Fold 1 | Val Accuracy: 0.7955
Epoch 17 | Fold 1 | Val Accuracy: 0.7986
Epoch 18 | Fold 1 | Val Accuracy: 0.8015
Epoch 19 | Fold 1 | Val Accuracy: 0.8041
Epoch 20 | Fold 1 | Val Accuracy: 0.8068
Epoch 21 | Fold 1 | Val Accuracy: 0.8091
Epoch 22 | Fold 1 | Val Accuracy: 0.8083
Epoch 23 | Fold 1 | Val Accuracy: 0.8086
Epoch 24 | Fold 1 | Val Accuracy: 0.8090


In [21]:

from sklearn.linear_model import LogisticRegression
# Stratified K-Fold setup for LR
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_overall_acc = 0
best_preds, best_labels = None, None

# Training loop with Stratified K-Fold for LR
fold = 1
for train_idx, val_idx in skf.split(X_scaled, y_cleaned):
    print(f"\n--- Fold {fold} ---")
    x_train_fold, x_val_fold = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train_fold, y_val_fold = y_cleaned.iloc[train_idx], y_cleaned.iloc[val_idx]

    # Handle class imbalance using SMOTE
    smote = SMOTE(random_state=42)
    x_train_res, y_train_res = smote.fit_resample(x_train_fold, y_train_fold)

    # Compute class weights for the training set
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_res), y=y_train_res)
    class_weights_dict = dict(zip(np.unique(y_train_res), class_weights))

    # Initialize and train Logistic Regression model
    model = LogisticRegression(max_iter=1000, class_weight=class_weights_dict, solver='lbfgs', multi_class='multinomial', random_state=42)
    model.fit(x_train_res, y_train_res)

    # Evaluate the model on the validation set
    val_preds = model.predict(x_val_fold)
    val_accuracy = np.mean(val_preds == y_val_fold)
    print(f"Validation Accuracy for Fold {fold}: {val_accuracy:.4f}")

    # Track best performing fold
    if val_accuracy > best_overall_acc:
        best_overall_acc = val_accuracy
        best_preds = val_preds
        best_labels = y_val_fold

    fold += 1

# Print best validation accuracy across all folds
print("\nBest Validation Accuracy across all folds:", round(best_overall_acc, 4))

# Print the classification report
print("\nClassification Report:")
# target_names = sorted(y_cleaned.unique().tolist())
target_names = [str(cls) for cls in le.classes_]
print(classification_report(best_labels, best_preds, target_names=target_names))



--- Fold 1 ---


C:\Users\dksri\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Validation Accuracy for Fold 1: 0.7255

--- Fold 2 ---


C:\Users\dksri\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Validation Accuracy for Fold 2: 0.7225

--- Fold 3 ---


C:\Users\dksri\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Validation Accuracy for Fold 3: 0.7137

Best Validation Accuracy across all folds: 0.7255

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.93      0.96       208
           1       0.50      0.59      0.54       458
           2       0.68      0.78      0.73       424
           3       0.98      0.90      0.94       134
           4       0.82      0.95      0.88       129
           5       0.72      0.76      0.74       483
           6       0.59      0.54      0.56       863
           7       0.99      0.99      0.99       192
           8       0.72      0.83      0.77       618
           9       0.70      0.75      0.73       841
          10       0.82      0.70      0.75      1649

    accuracy                           0.73      5999
   macro avg       0.77      0.79      0.78      5999
weighted avg       0.73      0.73      0.73      5999



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Stratified K-Fold setup for Random Forest
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_overall_acc = 0
best_preds, best_labels = None, None

# Training loop with Stratified K-Fold for Random Forest
fold = 1
for train_idx, val_idx in skf.split(X_scaled, y_cleaned):
    print(f"\n--- Fold {fold} ---")
    x_train_fold, x_val_fold = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train_fold, y_val_fold = y_cleaned.iloc[train_idx], y_cleaned.iloc[val_idx]

    # Handle class imbalance using SMOTE
    smote = SMOTE(random_state=42)
    x_train_res, y_train_res = smote.fit_resample(x_train_fold, y_train_fold)

    # Compute class weights for the training set
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_res), y=y_train_res)
    class_weights_dict = dict(zip(np.unique(y_train_res), class_weights))

    # Initialize and train Random Forest model
    model = RandomForestClassifier(n_estimators=100, class_weight=class_weights_dict, random_state=42)
    model.fit(x_train_res, y_train_res)

    # Evaluate the model on the validation set
    val_preds = model.predict(x_val_fold)
    val_accuracy = np.mean(val_preds == y_val_fold)
    print(f"Validation Accuracy for Fold {fold}: {val_accuracy:.4f}")

    # Track best performing fold
    if val_accuracy > best_overall_acc:
        best_overall_acc = val_accuracy
        best_preds = val_preds
        best_labels = y_val_fold

    fold += 1

# Print best validation accuracy across all folds
print("\nBest Validation Accuracy across all folds:", round(best_overall_acc, 4))

# Print the classification report
print("\nClassification Report:")
print(classification_report(best_labels, best_preds, target_names=target_names))

# Compute and print confusion matrix
# cm = confusion_matrix(best_labels, best_preds)
# print("\nConfusion Matrix:")
# print(cm)

# Optional: Visualize the confusion matrix using Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()



--- Fold 1 ---
